# <h1 style="text-align: center;"> Figure plotting notebook </h1>

This Python notebook loads MHWs netCDF files and produces figures.

## Setup

Run these cells only once to setup all the main necessary modules to generate the figures.

In [ ]:
# Enables modules autoreload (important during development)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# Advanced imports
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pymannkendall as mk

# Local imports
import balearic_mhws.config as config
import balearic_mhws.plotting.utils as utils

In [ ]:
from balearic_mhws.data.io import load_mhws, open_bathy

# Load the two MHW annual metrics dataset that were computed with REP and MEDREA respectively
ds_mhws_rep = load_mhws("yearly", "rep", False, "balears", (1987,2021))
ds_mhws_medrea = load_mhws("yearly", "medrea", False, "balears", (1987,2021))

# Load the MEDREA bathymetry dataset
ds_bathy_medrea = open_bathy()

# Interpolate bathymetry data onto REP grid
ds_bathy_rep = ds_bathy_medrea.interp(
    lat = ds_mhws_rep.lat,
    lon = ds_mhws_rep.lon,
    method = "linear"
)

In [ ]:
from balearic_mhws.plotting.utils import apply_regional_mask

# Compute regional masks
da_regions = xr.full_like(ds_mhws_medrea.isel(depth=0, year=0).total_days, np.nan)

continental_coast_mask = apply_regional_mask(da_regions, 'continental_coast', ds_bathy_medrea, return_mask=True)
balearic_coast_mask = apply_regional_mask(da_regions, 'balearic_coast', ds_bathy_medrea, return_mask=True)
balearic_sea_deep_mask = apply_regional_mask(da_regions, 'balearic_sea_deep', ds_bathy_medrea, return_mask=True)
west_algerian_deep_mask = apply_regional_mask(da_regions, 'west_algerian_deep', ds_bathy_medrea, return_mask=True)

# Compute a regional dataset with a number for each subregion
da_regions = da_regions.where(~continental_coast_mask, other=1)
da_regions = da_regions.where(~balearic_coast_mask, other=2)
da_regions = da_regions.where(~balearic_sea_deep_mask, other=3)
da_regions = da_regions.where(~west_algerian_deep_mask, other=4)

In [ ]:
from roman import toRoman

# Basic options
clim_period = (1987, 2021)
subplot_labels = [chr(ord('a')+n) + ')' for n in range(26)]
subplot_labels_roman = [toRoman(n) for n in range(101)]

## Figure 1

The **Figure 1** is an illustration of the region used, using some real data and a blueish color map.

In [ ]:
from balearic_mhws.plotting.plot import plot_map

# Select total days metric derived from REP in 2022
da = ds_mhws_rep.total_days.sel(year=2022)

# Plotting the annual metric as a map
plot_map(
    # Parameter of data
    da.lon, da.lat,
    da,
    
    # Parameter of figure
    figsize = (12,4),
    fontsize = 12,
    figdpi = 200,

    # Parameters of graph
    yticks = 4,
    bottom_labels = False,
    left_labels = False,

    # Parameter of colorbar
    add_cbar = False,
    cmap = 'cmo.tempo',
    vlim = (20, 250),

    show_plots = True,
)

## Figure 2

**Figure 2** is meant to illustrate the marine heatwaves definition. It uses the data of temperature in 2022 in the region.

In [ ]:
from balearic_mhws.data.io import load_mhws

# Loads the region-averaged MHW dataset derived from REP
ds_mhws_rep_mean = load_mhws(
    ds_type = 'all_events',
    dataset_used = 'rep_mean',
    detrended = False,
    region = 'balears',
    clim_period = clim_period
)

In [ ]:
from datetime import date
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import scipy.ndimage as ndimage

from balearic_mhws.processing.compute_mhws import get_mhw_ts_from_ds
from balearic_mhws.plotting.plot import get_locator_from_ticks


# The ticks of the y axis
yticks = (5,0)
yticks_minor = (2.5,)

# The maximum category to be shown
max_thr = 2

# Set the fontsize figure-wise
with plt.rc_context({'font.size': 12}):
    # Convert raw dataset to a more friendly format
    time, mhws = get_mhw_ts_from_ds(ds_mhws_rep_mean, None, None, None)

    # Create matplotlib fig and ax
    fig = plt.figure(figsize=(8.6, 5), dpi=300, constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1)

    # Plot the three main lines
    ax.plot(time, mhws["clim_sst"], label="REP-SST", c='k', lw=1)
    ax.plot(time, mhws["clim_thresh"], label="1987-2021 p90", c='r', ls='--', lw=1)
    ax.plot(time, mhws["clim_seas"], label="1987-2021 mean", c='r', lw=1)

    
    # Style for the 4 first categories
    colors = { 1: '#ffd86e', 2: '#ff621f', 3: '#df391b', 4: '#861a15'}
    lss = { 1: '--', 2: '-.', 3: ':', 4: '..'}

    # Compute thresholds for the categories
    for i in range(2, max_thr+2):
        mhws[f"clim_{i}thresh"] = mhws["clim_thresh"]*i - mhws["clim_seas"]*(i-1)
        ax.plot(time, mhws[f"clim_{i}thresh"], c="k", alpha=1, ls=lss[i], lw=1) 

    # Plot patches of color for each event and categories
    for ev in range(mhws["n_events"]):
        id_slice = slice(mhws["index_start"][ev], mhws["index_end"][ev]+1)
        temp = mhws["clim_sst"][id_slice]
        
        # Patches of color for category 1
        ax.fill_between(
            time[id_slice],
            mhws["clim_thresh"][id_slice],
            np.minimum(
                mhws["clim_sst"][id_slice],
                mhws["clim_2thresh"][id_slice],
            ),
            color='#ffd86e',
        )

        # Patches of color for category > 1
        for i in range(2, max_thr+1):
            lower_thr = mhws[f"clim_{i}thresh"][id_slice]
            upper_thr = mhws[f"clim_{i+1}thresh"][id_slice]
            times = time[id_slice]

            exceed_bool = temp - lower_thr
            exceed_bool = exceed_bool>=0

            events, n_events = ndimage.label(exceed_bool)
            
            for ev_ in range(1,n_events+1):
                slice_ = slice(np.where(events == ev_)[0][0]-1, np.where(events == ev_)[0][-1]+1)

                if times[np.where(events == ev_)[0][0]].astype('datetime64[Y]').astype(int) + 1970 == 2022 and times[np.where(events == ev_)[0][0]].astype('datetime64[M]').astype(int) % 12 + 1 == 12:
                    print(f"Filling Cat {i} from {times[np.where(events == ev_)[0][0]]} to {times[np.where(events == ev_)[0][-1]]}")

                ax.fill_between(
                    times[slice_],
                    lower_thr[slice_],
                    np.minimum(
                        temp[slice_],
                        upper_thr[slice_],
                    ),
                    color=colors[i],
                )

    # matplotlib options
    ax.grid(ls='--', alpha=0.5)
    ax.grid(which='minor', ls='--', alpha=0.3)
    ax.set_ylabel("SST [°C]")

    ax.set_xlim(date(2022, 1, 1), date(2022, 12, 31))
    
    ax.set_xticks(
        [date(2022, mm, 1) for mm in range(1, 13, 2)],
        ['Jan', 'Mar', 'May', 'Jul', 'Sep', 'Nov'],
    )
    ax.yaxis.set_major_locator(get_locator_from_ticks(yticks))
    
    ax.get_legend_handles_labels()

    ax.legend(
        [
            Patch(color='#ff621f'),
            Line2D([0], [0], c='k', ls='-.'),
            Patch(color='#ffd86e'),
            Line2D([0], [0], c='r', ls='--'),
            Line2D([0], [0], c='r'),
            Line2D([0], [0], color='k'),
        ], ["MHW category II", "Category thresholds", "MHW category I", "90$\mathregular{^{th}}$ percentile clim.", "Mean clim.", "Observed SST"],
        loc='upper left',
    )

plt.show()


## Figure 3

**Figure 3** is meant to illustrate the region and subregions considered in this study. It consists of two map of the Mediterranean Sea and the Balearic Islands region, respectively.

Note that in the report, the figure has been modified in an external software.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import cartopy.crs as ccrs

from balearic_mhws.plotting.plot import plot_map


# Generate color map
alpha = 0.9
cmap = ListedColormap([('C0', alpha), ('C1', alpha), ('C2', alpha), ('C3', alpha)])

# Plot Mediterranean Sea map
fig, ax = plot_map(
    # Parameter of data
    da_regions.lon, da_regions.lat,
    da_regions,

    # Parameter of figure
    figsize = (12, 4),
    fontsize = 12,
    figdpi = 200,

    # Parameters of graph
    extent = [-5.7, 16.4, 34.6, 45],
    yticks = 4,
    xticks = 5,

    # Parameter of colorbar
    add_cbar = False,
    cmap = cmap,
    vlim = (1, 4),
)

# Create a zoom of first map
x1, x2, y1, y2 = -1, 5, 37.7, 41
axins = ax.inset_axes(
    [1.02, 0.05, 0.9, 0.9],
    xlim=(x1, x2), ylim=(y1, y2), xticklabels=[], yticklabels=[], projection=ccrs.PlateCarree()
)

# Plot Balearic Islands region map
plot_map(
    # Parameter of data
    da_regions.lon, da_regions.lat,
    da_regions,

    # Parameter of figure
    fig = fig, ax = axins,
    figsize = (8, 20),
    fontsize = 12,
    figdpi = 200,

    # Parameter of colorbar
    add_cbar = False,
    cmap = cmap,
    vlim = (1, 4),

    # Parameters of graph
    extent = None,
    yticks = 4,
    legend = {
        'handles': [
            Patch(color='C0'),
            Patch(color='C1'),
            Patch(color='C2'),
            Patch(color='C3'),
        ],
        'labels': ["SPC", "BIC", "BS", "NWA"],
        'loc': 'lower right',
        'handlelength': 1.2,
    },
)

plt.show()

## Table 2

**Table 2** gives the mean, STD and trend for several models and periods, averaging the annual metrics over the region.

In [ ]:
from scipy.stats import theilslopes

# Iterate over stats
for i, stat in enumerate([
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]):
    print(f"{i+1}. {config.mhws_stats_shortname[stat]}")

    # Iterate over models/periods
    for model in ["REP82", "REP", "MEDREA"]:
        if model == "REP82":
            da = ds_mhws_rep[stat].sel(year=slice(1982,2023))
        elif model == "REP":
            da = ds_mhws_rep[stat].sel(year=slice(1987,2022))
        elif model == "MEDREA":
            da = ds_mhws_medrea[stat].sel(depth=0, method='nearest')

        # Spatial averaging over the region
        y = da.mean(dim=["lon", "lat"])

        # Period averaging over 1982-2023 or 1987-2022
        # Using xarray functions
        mean = y.mean(dim='year')
        std = y.std(dim='year')

        print(f"{model} mean : {mean:.2f} ± {std:.2f} {config.mhws_stats_units[stat]}")

        # Calculation of trend over 1982-2023 or 1987-2022
        # Using Hamed Rao modified Mann-Kendall test for assessing significance of trend
        # Using Theil Sen estimator for trend slope
        # Using pymannkendall package to perform trend calculation
        trend, h, p, z, tau, s, var_s, slope, intercept = mk.hamed_rao_modification_test(y, alpha=0.05)

        # To calculate confidence interval (CI), needs to use scipy because pymannkendall
        # package doesn't include it
        slope_, intercept_, lower, upper = theilslopes(y, alpha=0.05)

        # h variable assesses the significance of the trend
        if h:
            print(f"{model} trend : {slope*10:.2f} ± {(upper-lower)/2 * 10:.2f} {config.mhws_stats_units[stat]}/decade (p-value={p:.5f})")
        
        else:
            print(f"No significant trend for {model}.")

    
    print()

## Figure 4

**Figure 4** shows maps of means and trends of MHW annual metrics, as well as the values in 2022

In [ ]:
# Apply the Mann-Kendall test on the whole REP dataset
h, p, slope = xr.apply_ufunc(
    utils.apply_mk_test,
    ds_mhws_rep[config.mhws_basic_stats],
    input_core_dims = [["year"]],
    output_core_dims = [[], [], []],
    vectorize = True,
    dask = "parallelized",
    output_dtypes = [bool, float, float],
)

# Final dataset with decadal trends
ds_trend = slope.where(h) * 10

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps


# Parameters to plot in the figure
columns = ['Mean', '2022', 'Trend']
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]

# Options
names = {
    "total_days":           r"$\bf{Total}$" "\n" r"$\bf{days}$ [days]",
    "duration":             r"$\bf{Mean}$" "\n" r"$\bf{duration}$ [days]",
    "total_icum":           r"$\bf{Cumulative}$" "\n" r"$\bf{intensity}$" "\n[°C.days]",
    "intensity_max_max":    r"$\bf{Maximum}$" "\n" r"$\bf{intensity}$ [°C]",
    "intensity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{intensity}$ [°C]",
    "severity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{severity}$",
}

# Size of the subplot grid
ncols = len(columns)
nrows = len(stats)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats and columns
for iss, stat in enumerate(stats):
    for ic, column in enumerate(columns):
        row = iss
        col = ic

        cmap = mhws_stats_cmaps[stat]

        if column == 'Mean':
            da = ds_mhws_rep[stat].mean(dim='year')
        
        elif column == '2022':
            da = ds_mhws_rep[stat].sel(year=2022)

        elif column == 'Trend':
            da = ds_trend[stat]
            cmap = 'cmo.speed'

        # Add the settings for the specific subplot
        subplots_settings.append(dict(
            # Parameter of subplot
            pos = iss*ncols + ic + 1,
            func = plot_map,

            # Parameter of data
            lon = da.lon,
            lat = da.lat,
            data = da,

            # Parameter of text
            title = r"$\bf{" + column.replace(' ', ' \ ') + "}$" if row == 0 else None,
            ylabel = names[stat] if col == 0 else None,
            fontsize_title = 1.2,
            ylabel_pad = -0.3,

            texts = [dict(
                x=0.02,
                y=0.96,
                s=utils.bold(subplot_labels[row + nrows*col]),
                ha='left',
                va='top',
                fontsize=18
            )],

            # Parameter of colorbar
            cmap = cmap,
            cbar_shrink = 0.6,
            cbar_ticks = 3,

            # Parameter of contours
            contours_levels = [2.5, 3.5],
            contours_data = [
                da_regions.lon,
                da_regions.lat,
                da_regions
            ],
            contours_kwargs = dict(
                colors='k', linewidths=1, zorder=-1
            ),

            # Parameters of graph
            xticks = 3,
            yticks = 4,
            left_labels = (col == 0),
            bottom_labels = (row == nrows - 1),
        ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    pad_subplots = (None, 0.2),
    
    # Parameter of figure
    figsize = (12,16),
    fig_fontsize = 13,
    figdpi = 200,

    # Must specify if the plot is a map
    fig_is_a_map = True,

    show_plots = True,
)

## Figure 5

**Figure 5** represents the timeseries of the annual metrics, averaged over the four subregions.

In [ ]:
import math

from balearic_mhws.plotting.plot import plot_timeserie, subplot
from balearic_mhws.plotting.utils import apply_regional_mask


# Parameters to plot in the figure
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]

# Size of the subplot grid
ncols = 2
nrows = math.ceil(len(stats) / ncols)

# Create subregional datasets
subregional_ds = {}

for region in regions:
    subregional_ds[region] = apply_regional_mask(ds_mhws_rep, region, ds_bathy_rep)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats
for id, stat in enumerate(stats):
    col = id // nrows
    row = id % nrows

    escaped_name = config.mhws_stats_shortname[stat].replace(' ', '\ ')

    # Add the settings for the specific subplot
    subplots_settings.append(dict(
        # Parameter of subplot
        pos = row * ncols + col + 1,
        func = plot_timeserie,

        # Parameter of data
        vars = {
            config.region_shortname[region]:
                subregional_ds[region][stat].mean(dim=["lon","lat"])
            
            for region in regions
        },
        times = subregional_ds[regions[0]].year,
        colors = {
            "Continental coast": 'C0',
            "Balearic Islands coast": 'C1',
            "Balearic Sea deep": 'C2',
            "West Algerian Basin deep": 'C3',
        },

        # Parameter of text
        title = fr"$\bf{{{escaped_name}}}${f' [{config.mhws_stats_units[stat]}]' if stat != 'severity_mean_byday' else ''}",
        legend = {
            'labels': [
                f"{subregional_ds[region][stat].mean(dim=['lon','lat']).mean(dim='year'):.2f}"
                for region in regions
            ],
            'ncol': 4,
            'loc': 4 if stat in ['intensity_max_max', 'intensity_mean_byday', 'severity_mean_byday'] else None,
            'handletextpad': 0.3,
            'columnspacing': 0.8,
            'handlelength': 1,
        },

        texts = [dict(
            x=0.06,
            y=1.03,
            s=utils.bold(subplot_labels[row + nrows*col]),
            ha='right',
            va='bottom',
            fontsize=18
        )],

        # Parameter of graph
        nans_to_zero = True,

        xticks = (10,),
        xticks_minor = 2,
        yticks = 3,

        bottom_labels = row == nrows-1,
    ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,

    # Parameter of figure
    figsize = (12, 5.4),
    figdpi = 200,
    fig_fontsize = 14,

    show_plots = True,
)

## Figure 6

**Figure 6** shows the mean and trend of the annual metrics for every depth levels, averaged over the region.

In [ ]:
from textwrap import wrap

from balearic_mhws.plotting.plot import plot_bars, subplot


# Parameters to plot in the figure
columns = [
    'Mean',
    'Trend'
]
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]

# Options
colors = {
    "total_days":           "#13847b",
    "duration":             "#e15f52",
    "total_icum":           "#ca8f18",
    "intensity_max_max":    "#af2f24",
    'intensity_mean_byday': "#af2f24",
    'severity_mean_byday':  "#c96716",
}

# Size of the subplot grid
ncols = len(columns)
nrows = len(stats)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats and columns
for row, stat in enumerate(stats):
    for col, column in enumerate(columns):
        ds = ds_mhws_medrea

        std, hatchs = None, None

        if column == 'Mean':
            da = ds[stat].mean(dim=['lon', 'lat']).mean(dim='year')
            std = ds[stat].mean(dim=['lon', 'lat']).std(dim='year')

        elif column == 'Trend':
            da = ds[stat].mean(dim=['lon', 'lat'])

            h, p, slope = xr.apply_ufunc(
                utils.apply_mk_test,
                da,
                input_core_dims=[["year"]],
                output_core_dims=[[], [], []],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[bool, float, float],
            )

            # Change from yearly trend to decadly trend
            da = slope.where(h, 0) * 10
        
        # Add the settings for the specific subplot
        subplots_settings.append(dict(
            # Parameter of subplot
            pos = row*ncols + col + 1,
            func = plot_bars,

            # Parameter of data
            depths = da.depth,
            vars = {'a': da},
            stds = {'a': std} if column == 'Mean' else None,
            colors = colors[stat],

            # Parameter of text
            title = utils.bold(column) if row == 0 else None,
            ylabel = utils.bold('\n'.join(wrap(config.mhws_stats_shortname[stat], 15))) + 
                        ('' if stat == 'severity_mean_byday' else f" [{config.mhws_stats_units[stat]}]")
                        if col == 0 else None,

            texts = [dict(
                x=0.98,
                y=(0.04-0.96) * [0,0,1,1,1,1,0,0,1,1,1,1][row + nrows*col] + 0.96,
                s=utils.bold(subplot_labels[row + nrows*col]),
                ha='right',
                va='bottom' if [0,0,1,1,1,1,0,0,1,1,1,1][row + nrows*col] else 'top',
                fontsize=18
            )],

            # Parameters of graph
            bars_pad = 0.5,

            xticks = 5,
            yticks = 2,

            left_labels = (col == 0),
        ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    
    # Parameter of figure
    figsize = (12, 15.2),
    fig_fontsize = 14,
    figdpi = 200,

    show_plots = True,
)

## Figures 7 & 8

**Figure 7 & 8** represents the timeseries of the annual metrics, averaged over the four subregions, for all depth levels.

In [ ]:
from roman import toRoman

from balearic_mhws.plotting.plot import plot_timeserie, subplot
from balearic_mhws.plotting.utils import apply_regional_mask


# Parameters to plot in the figure
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]


# Create subregional datasets
subregional_ds = {}

for region in regions:
    subregional_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)


# Iterate over two subsets of stats
for stats in [[
    'total_days',
    'duration',
    'total_icum',
],[
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]]:
    # Size of the subplot grid
    ncols = len(stats)
    nrows = len(depths)
    

    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over stats and depths
    for iss, stat in enumerate(stats):
        for id, depth in enumerate(depths):
            col = iss
            row = id
            
            # Choose the position of the subplot label
            if stat in ['intensity_max_max', 'intensity_mean_byday', 'severity_mean_byday']:
                subplot_label_ha = 'right'

                if depth > 190 and depth < 210:
                    subplot_label_va = 'top'
                else:
                    subplot_label_va = 'bottom'
            
            else:
                subplot_label_ha = 'left'
                subplot_label_va = 'top'

            # Add the settings for the specific subplot 
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row * ncols + col + 1,
                func = plot_timeserie,

                # Parameter of data
                vars = {
                    config.region_shortname[region]:
                        subregional_ds[region][stat].sel(depth=depth, method='nearest').mean(dim=["lon","lat"], keep_attrs=True)
                    
                    for region in (
                        regions[2:] if depth > 200 else regions
                    )
                },
                times = subregional_ds[regions[0]].year,
                colors = {
                    "Continental coast": 'C0',
                    "Balearic Islands coast": 'C1',
                    "Balearic Sea deep": 'C2',
                    "West Algerian Basin deep": 'C3',
                },

                # Parameter of text
                title = utils.bold(config.mhws_stats_shortname[stat]) + 
                        ('' if stat == 'severity_mean_byday' else f" [{config.mhws_stats_units[stat]}]")
                        if row == 0 else None,
                ylabel = fr"$\bf{{{depth:.0f} \ m}}$" if col == 0 else None,

                texts = [dict(
                    x=(0.02-0.98) * (subplot_label_ha == 'left') + 0.98,
                    y=(0.04-0.96) * (subplot_label_va == 'bottom') + 0.96,
                    s=utils.bold(subplot_labels_roman[row + nrows*col + 1].lower() + ')'),
                    ha=subplot_label_ha,
                    va=subplot_label_va,
                    fontsize=18
                )],

                # Parameters of graph
                legend = False,
                nans_to_zero = True,

                xticks = (10,),
                xticks_minor = 2,
                yticks = 3,

                bottom_labels = (row == nrows-1),
            ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,

        # Parameter of figure
        figsize = (12, 12),
        figdpi = 200,
        fig_fontsize = 14,

        show_plots = True,
    )

## Figure 9

**Figure 9** shows the means and trends of the annual metrics for every depth levels and subregions.

In [ ]:
from balearic_mhws.plotting.plot import plot_bars, subplot
from balearic_mhws.plotting.utils import apply_regional_mask


# Parameters to plot in the figure
columns = [
    'Mean',
    'Trend'
]
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]

# Options
names = {
    "total_days":           r"$\bf{Total}$" "\n" r"$\bf{days}$ [days]",
    "duration":             r"$\bf{Mean}$" "\n" r"$\bf{duration}$ [days]",
    "total_icum":           r"$\bf{Cumulative}$" "\n" r"$\bf{intensity}$" "\n[°C.days]",
    "intensity_max_max":    r"$\bf{Maximum}$" "\n" r"$\bf{intensity}$ [°C]",
    "intensity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{intensity}$ [°C]",
    "severity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{severity}$",
}
subplot_label_pos = {
    "total_days": 1,
    "duration": 1,
    "total_icum": 0,
    "intensity_max_max": 0,
    "intensity_mean_byday": 0,
    "severity_mean_byday": 0,
}


# Create subregional datasets
subregional_ds = {}

for region in regions:
    subregional_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)

# Size of the subplot grid
ncols = len(columns)
nrows = len(stats)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats and columns
for row, stat in enumerate(stats):
    for col, column in enumerate(columns):
        region_das = {}

        for region in regions:
            ds = subregional_ds[region].sel(depth=[0, 100, 200, 700, 1500], method='nearest')

            if column == 'Mean':
                da = ds[stat].mean(dim=['lon', 'lat']).mean(dim='year')
            
            elif column == '2022':
                da = ds[stat].sel(year=2022).mean(dim=['lon', 'lat'])
     
            elif column == 'Trend':
                da = ds[stat].mean(dim=['lon', 'lat'])

                def apply_mk_test(y):
                    if np.isnan(y).all():
                        return np.nan, np.nan, np.nan
                    
                    trend, h, p, z, tau, s, var_s, slope, intercept = mk.hamed_rao_modification_test(y, alpha=0.05)
                    
                    return h, p, slope

                h, p, slope = xr.apply_ufunc(
                    apply_mk_test,
                    da,
                    input_core_dims=[["year"]],
                    output_core_dims=[[], [], []],
                    vectorize=True,
                    dask="parallelized",
                    output_dtypes=[bool, float, float],
                )

                da = slope.where(h, 0)

                # Change from yearly trend to decadly trend
                da = da * 10

                unit = f"[{config.mhws_stats_units[stat]} /decade]"
            
            region_das[region] = da


        # Add the settings for the specific subplot
        subplots_settings.append(dict(
            # Parameter of subplot
            pos = row*ncols + col + 1,
            func = plot_bars,

            # Parameter of data
            depths = da.depth.values,
            vars = region_das,

            # Parameter of text
            title = utils.bold(column) if row == 0 else None,
            ylabel = names[stat] if col == 0 else None,

            texts = [dict(
                x=0.98,
                y=(0.96-0.04) * subplot_label_pos[stat] + 0.04,
                s=utils.bold(subplot_labels[row + nrows*col]),
                ha='right',
                va='top' if subplot_label_pos[stat] else 'bottom',
                fontsize=18
            )],

            # Parameters of graph
            bars_pad = 1.2,
            xticks = 5,
            left_labels = (col == 0),
        ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    
    # Parameter of figure
    figsize = (12, 12),
    fig_fontsize = 14,
    figdpi = 200,

    show_plots = True,
)

## Figure 10

**Figure 10** highlights MHW activity that has been linked to mesoscale activity in 1998, 2008 and 2017. It consists of maps of annual metrics for those given years for every depth level.

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
stat = 'total_days'
years = [1998, 2008, 2017]
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]

# Size of the subplot grid
ncols = len(years)
nrows = len(depths)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats and columns
for id, depth in enumerate(depths):
    for iy, year in enumerate(years):
        row = id
        col = iy

        ds = ds_mhws_medrea.sel(year=year, depth=depth, method='nearest')

        # Add the settings for the specific subplot 
        subplots_settings.append(dict(
            # Parameter of subplot
            pos = row * ncols + col + 1,
            func = plot_map,

            # Parameter of data
            lon = ds.lon,
            lat = ds.lat,
            data = ds[stat],

            # Parameter of text
            title = utils.bold(year) if row == 0 else None,
            ylabel = utils.bold(f"{depth:.0f} m") if col == 0 else None,
            ylabel_pad = -0.02,

            # Parameter of graph
            zero_to_nan = True,

            left_labels = False,
            bottom_labels = False,
            xticks = 6,
            yticks = 4,
        ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    
    # Parameter of figure
    figsize = (5,12.8),
    fig_fontsize = 14,
    figdpi = 200,

    # Parameter of colorbar
    fig_cbar = True,
    fig_cbar_unit = utils.bold(config.mhws_stats_shortname[stat]) + f" [{config.mhws_stats_units[stat]}]",
    fig_cbar_fraction = 0.01,
    fig_cbar_pad = 0.005,
    fig_cbar_orientation='horizontal',
    fig_cbar_ticks = 4,
    fig_cmap = mhws_stats_cmaps[stat],

    # Must specify if the plot is a map
    fig_is_a_map = True,

    show_plots = True,
)

## Figure A1&2

**Figure A1 & A2** consist of maps of total days and maximum intensity, respectively, for every years and depth level.

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
years = range(1987, 2023)
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]

# Iterate over stats
for stat in [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]:
    # Size of the subplot grid
    ncols = len(years)
    nrows = len(depths)


    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over depths and years
    for id, depth in enumerate(depths):
        for iy, year in enumerate(years):
            row = id
            col = iy

            ds = ds_mhws_medrea.sel(year=year, depth=depth, method='nearest')

            # Add the settings for the specific subplot 
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row * ncols + col + 1,
                func = plot_map,

                # Parameter of data
                lon = ds.lon,
                lat = ds.lat,
                data = ds[stat],

                # Parameter of text
                title = year if row == 0 and (year % 2 == 0) else None,
                ylabel = f"{depth:.0f}m" if col == 0 and row % 2 == 0 else None,
                ylabel_pad=-0.08,

                # Parameter of graph
                zero_to_nan = True,

                left_labels = False,
                bottom_labels = False,
                xticks = 6,
                yticks = 4,
            ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,
        
        # Parameter of figure
        subplotsize = (6.6, 5),
        figdpi = 30,
        fig_fontsize = 120,
        pad_subplots = (0.05, 0.05),

        # Parameter of colorbar
        fig_cbar = True,
        fig_cbar_unit = f"[{config.mhws_stats_units[stat]}]",
        fig_cbar_fraction = 0.05,
        fig_cbar_pad = 0.01,
        fig_cbar_orientation='horizontal',
        fig_cbar_ticks = 5,
        fig_cmap = mhws_stats_cmaps[stat],

        # Must specify if the plot is a map
        fig_is_a_map = True,

        show_plots = True,
    )

## Figure A3

**Figure A3** is meant to illustrate the evolution of temperature for every depth. It consists of temperature time series derived from MEDREA as well as the mean and 90th percentile climatology, at every depth levels.

In [ ]:
from balearic_mhws.data.io import load_mhws

# Loads the region-averaged MHW dataset derived from MEDREA
ds_mhws_medrea_mean = load_mhws(
    ds_type = 'all_events',
    dataset_used = 'medrea_mean',
    detrended = False,
    region = 'balears',
    clim_period = clim_period
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import date

from balearic_mhws.processing.compute_mhws import get_mhw_ts_from_ds
from balearic_mhws.plotting.plot import get_locator_from_ticks


# Parameters to plot in the figure
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]
date_ranges = [(0.5, 39.31), (0.5, 39.31)]

# Set the fontsize figure-wise
with plt.rc_context({'font.size': 12}):
    # Create matplotlib fig
    fig = plt.figure(figsize=(10, 12), dpi=200, constrained_layout=True)

    # Size of the subplot grid
    nrows = len(depths)
    ncols = len(date_ranges)+1

    # Iterate over dates and depths
    for col, (t0, t1) in enumerate(date_ranges):
        for row, depth in enumerate(depths):
            # Convert raw dataset to a more friendly format
            time, mhws = get_mhw_ts_from_ds(ds_mhws_medrea_mean, None, None, depth)

            # If there is no MHW event, let's skip it
            if mhws == -1:
                continue

            index = (row*ncols + 1, row*ncols + 2) if col == 0 else row*ncols + col + 2

            # Create matplotlib ax
            ax = fig.add_subplot(nrows, ncols, index)

            # Plot the three main lines 
            ax.plot(time, mhws["clim_sst"], label="thetao", c='k', lw=1)
            ax.plot(time, mhws["clim_thresh"], label="1987-2021 p90", c='r', ls='--', lw=1)
            ax.plot(time, mhws["clim_seas"], label="1987-2021 mean", c='r', lw=1)

            # Compute thresholds for the categories
            mhws["clim_2thresh"] = mhws["clim_thresh"]*2 - mhws["clim_seas"]
            mhws["clim_3thresh"] = mhws["clim_thresh"]*3 - mhws["clim_seas"]*2

            # Plot patches of color for each event and categories
            for ev in range(mhws["n_events"]):
                id_slice = slice(mhws["index_start"][ev], mhws["index_end"][ev]+1)

                # Patches of color for category 1
                ax.fill_between(
                    time[id_slice],
                    mhws["clim_thresh"][id_slice],
                    np.minimum(
                        mhws["clim_sst"][id_slice],
                        mhws["clim_2thresh"][id_slice],
                    ),
                    color='#ffd86e',
                )
                
                # Patches of color for category 2
                strong_intensity = mhws["clim_sst"][id_slice]
                strong_intensity[np.where(mhws["clim_sst"][id_slice] < mhws["clim_2thresh"][id_slice])] = np.nan

                ax.fill_between(
                    time[id_slice],
                    mhws["clim_2thresh"][id_slice],
                    np.minimum(
                        strong_intensity,
                        mhws["clim_3thresh"][id_slice],
                    ),
                    color='#ff621f',
                )
                
                # Patches of color for category 3
                severe_intensity = mhws["clim_sst"][id_slice]
                severe_intensity[np.where(mhws["clim_sst"][id_slice] < mhws["clim_3thresh"][id_slice])] = np.nan

                ax.fill_between(
                    time[id_slice],
                    mhws["clim_3thresh"][id_slice],
                    severe_intensity,
                    color='#df391b',
                )

            # matplotlib options
            ax.grid(True, ls='--', alpha=0.5)
            
            ax.yaxis.set_major_locator(get_locator_from_ticks(1))

            if col == 0:
                ax.set_ylabel(fr"$\bf{{{depth:.0f}m}}$")
            else:
                ax.tick_params(which='both', left=False, labelleft=False)
            
            if col == 1:
                ax.set_xlim(date(2020, 1, 1), date(2022, 12, 31))
                ax.set_xticks(
                    [date(2021, 1, 1), date(2022, 1, 1)],
                    ["2021", "2022"]
                )

            else:
                ax.set_xlim(date(1987, 1, 1), date(2022, 12, 31))
                ax.set_xticks(
                    [date(1990, 1, 1), date(2000, 1, 1), date(2010, 1, 1), date(2020, 1, 1)],
                    ["1990", "2000", "2010", "2020"]
                )
                
            if not row == nrows-1:
                ax.tick_params(which='both', bottom=False, labelbottom=False)

            fig.align_ylabels()

plt.show()